*Made by Phuc*

## MỤC TIÊU ##

1. Đọc file `data/processed/olap_sales_view.csv`
2. Tính Iceberg Cube bằng thuật toán BUC top-down
3. Phân cụm khách hàng bằng RFM và K-Means
4. Xuất file cho app và báo cáo

In [ ]:
# Import thư viện cần dùng
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

## 1. Cấu hình đường dẫn và tham số

Input sau bước tiền xử lý:

`data/processed/olap_sales_view.csv`

Output của phần này là:

- `outputs/iceberg_cube.csv`
- `outputs/customer_clusters.csv`
- `outputs/model_metrics.json`

In [ ]:
# Đường dẫn input/output
INPUT_PATH = Path("data/processed/olap_sales_view.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ICEBERG_OUTPUT_PATH = OUTPUT_DIR / "iceberg_cube.csv"
CLUSTER_OUTPUT_PATH = OUTPUT_DIR / "customer_clusters.csv"
METRICS_OUTPUT_PATH = OUTPUT_DIR / "model_metrics.json"

# Dimensions dùng cho BUC Iceberg Cube
BUC_DIMENSIONS = [
    "order_month",
    "customer_state",
    "product_category_name",
    "payment_type",
]

# Điều kiện iceberg: chỉ giữ cell có số dòng fact >= MIN_SUP
MIN_SUP = 50

# Features dùng cho phân cụm khách hàng
CLUSTER_FEATURES = [
    "recency",
    "frequency",
    "monetary",
    "avg_order_value",
    "avg_review_score",
    "avg_delivery_days",
]

K_RANGE = range(2, 8)
RANDOM_STATE = 42

## 2. Đọc dữ liệu

File input cần có các cột tối thiểu:

- `order_id`
- `customer_unique_id`
- `order_purchase_timestamp`
- `order_month`
- `customer_state`
- `product_category_name`
- `payment_type`
- `total_amount`
- `review_score`
- `delivery_days`

In [ ]:
def load_olap_view(path: Path) -> pd.DataFrame:
    """Đọc dữ liệu OLAP view và kiểm tra các cột cần có."""
    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file {path}. Vui lòng kiểm tra lại đường dẫn."
        )

    df = pd.read_csv(path)

    required_columns = [
        "order_id",
        "customer_unique_id",
        "order_purchase_timestamp",
        "order_month",
        "customer_state",
        "product_category_name",
        "payment_type",
        "total_amount",
        "review_score",
        "delivery_days",
    ]

    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"File input đang thiếu các cột: {missing_columns}")

    return df


df = load_olap_view(INPUT_PATH)
df.head()

In [ ]:
# Kiểm tra nhanh dữ liệu
df.info()

## 3. Chuẩn bị dữ liệu cho BUC

BUC dùng `support_count` để pruning. Vì `count` có tính giảm dần khi đi sâu xuống các dimension con, nếu một nhóm không đạt `min_sup` thì các nhóm con của nó cũng không cần xét tiếp.

In [ ]:
def prepare_cube_data(df: pd.DataFrame, dimensions: list[str]) -> pd.DataFrame:
    """Làm sạch đơn giản các cột dùng cho BUC."""
    cube_df = df.copy()

    for dim in dimensions:
        cube_df[dim] = cube_df[dim].fillna("unknown").astype(str)

    numeric_columns = ["total_amount", "review_score", "delivery_days"]
    for col in numeric_columns:
        cube_df[col] = pd.to_numeric(cube_df[col], errors="coerce")

    cube_df["total_amount"] = cube_df["total_amount"].fillna(0)
    cube_df["review_score"] = cube_df["review_score"].fillna(cube_df["review_score"].median())
    cube_df["delivery_days"] = cube_df["delivery_days"].fillna(cube_df["delivery_days"].median())

    return cube_df


cube_df = prepare_cube_data(df, BUC_DIMENSIONS)
cube_df[BUC_DIMENSIONS + ["total_amount", "review_score", "delivery_days"]].head()

## 4. Thuật toán BUC Top-Down Iceberg Cube

Cách hoạt động:

1. Bắt đầu từ cell tổng quát `ALL`.
2. Chia dữ liệu theo từng dimension.
3. Nếu một partition có `support_count < min_sup`, dừng nhánh đó.
4. Nếu partition đạt `min_sup`, lưu cell và tiếp tục đi sâu xuống dimension sau.
5. Kết quả cuối cùng chỉ gồm các cube cell đạt điều kiện iceberg.

In [ ]:
def aggregate_cube_cell(data: pd.DataFrame, prefix: dict, dimensions: list[str]) -> dict:
    """Tạo một cube cell từ partition hiện tại."""
    record = {}

    for dim in dimensions:
        record[dim] = prefix.get(dim, "ALL")

    record["cuboid_level"] = sum(1 for dim in dimensions if record[dim] != "ALL")
    record["support_count"] = int(len(data))
    record["order_count"] = int(data["order_id"].nunique())
    record["total_sales"] = float(data["total_amount"].sum())
    record["avg_review_score"] = float(data["review_score"].mean())
    record["avg_delivery_days"] = float(data["delivery_days"].mean())

    return record


def buc_top_down(
    data: pd.DataFrame,
    dimensions: list[str],
    min_sup: int,
    start_dim: int = 0,
    prefix: dict | None = None,
    results: list | None = None,
) -> list[dict]:
    """
    Tính Iceberg Cube bằng cơ chế BUC top-down.

    data: partition hiện tại
    dimensions: danh sách dimension
    min_sup: ngưỡng support tối thiểu
    start_dim: dimension bắt đầu xét
    prefix: các giá trị dimension đã chọn
    results: danh sách cube cell đạt điều kiện iceberg
    """
    if prefix is None:
        prefix = {}
    if results is None:
        results = []

    # Iceberg pruning
    if len(data) < min_sup:
        return results

    # Lưu cell hiện tại
    results.append(aggregate_cube_cell(data, prefix, dimensions))

    # Đi xuống các dimension con
    for dim_index in range(start_dim, len(dimensions)):
        dim = dimensions[dim_index]

        for value, partition in data.groupby(dim, dropna=False):
            if len(partition) >= min_sup:
                prefix[dim] = value
                buc_top_down(
                    data=partition,
                    dimensions=dimensions,
                    min_sup=min_sup,
                    start_dim=dim_index + 1,
                    prefix=prefix,
                    results=results,
                )
                prefix.pop(dim, None)

    return results


buc_results = buc_top_down(
    data=cube_df,
    dimensions=BUC_DIMENSIONS,
    min_sup=MIN_SUP,
)

iceberg_cube = pd.DataFrame(buc_results)

iceberg_cube = iceberg_cube.sort_values(
    by=["cuboid_level", "support_count", "total_sales"],
    ascending=[True, False, False],
).reset_index(drop=True)

iceberg_cube.to_csv(ICEBERG_OUTPUT_PATH, index=False)

print(f"Đã xuất file: {ICEBERG_OUTPUT_PATH}")
print(f"Số cube cell đạt điều kiện iceberg: {len(iceberg_cube)}")
iceberg_cube.head(20)

In [ ]:
# Xem các nhóm chi tiết có doanh thu cao nhất
iceberg_cube.sort_values("total_sales", ascending=False).head(20)

## 5. Tạo dữ liệu RFM cho phân cụm

RFM gồm:

- `Recency`: khách hàng mua gần đây hay đã lâu.
- `Frequency`: số lần mua hàng.
- `Monetary`: tổng số tiền khách hàng đã chi.

Có thêm một số biến hỗ trợ như `avg_order_value`, `avg_review_score`, `avg_delivery_days`.

In [ ]:
def build_customer_rfm(df: pd.DataFrame) -> pd.DataFrame:
    """Tạo bảng RFM ở cấp khách hàng."""
    rfm_df = df.copy()

    rfm_df["order_purchase_timestamp"] = pd.to_datetime(
        rfm_df["order_purchase_timestamp"], errors="coerce"
    )

    rfm_df = rfm_df.dropna(subset=["order_purchase_timestamp", "customer_unique_id"])

    numeric_columns = ["total_amount", "review_score", "delivery_days"]
    for col in numeric_columns:
        rfm_df[col] = pd.to_numeric(rfm_df[col], errors="coerce")

    rfm_df["total_amount"] = rfm_df["total_amount"].fillna(0)
    rfm_df["review_score"] = rfm_df["review_score"].fillna(rfm_df["review_score"].median())
    rfm_df["delivery_days"] = rfm_df["delivery_days"].fillna(rfm_df["delivery_days"].median())

    snapshot_date = rfm_df["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

    customer_rfm = (
        rfm_df.groupby("customer_unique_id")
        .agg(
            recency=("order_purchase_timestamp", lambda x: (snapshot_date - x.max()).days),
            frequency=("order_id", "nunique"),
            monetary=("total_amount", "sum"),
            avg_order_value=("total_amount", "mean"),
            avg_review_score=("review_score", "mean"),
            avg_delivery_days=("delivery_days", "mean"),
        )
        .reset_index()
    )

    return customer_rfm


customer_rfm = build_customer_rfm(df)
customer_rfm.head()

In [ ]:
customer_rfm[CLUSTER_FEATURES].describe()

## 6. Chọn số cụm bằng Silhouette Score

Vì K-Means dùng khoảng cách, dữ liệu cần được chuẩn hóa trước khi phân cụm.

In [ ]:
def prepare_clustering_matrix(customer_rfm: pd.DataFrame, features: list[str]) -> tuple[np.ndarray, pd.DataFrame, StandardScaler]:
    """Chuẩn bị ma trận dữ liệu cho K-Means."""
    X = customer_rfm[features].copy()
    X = X.fillna(X.median(numeric_only=True))

    # Giảm lệch cho các biến thường bị skew trong RFM
    skewed_columns = ["frequency", "monetary", "avg_order_value"]
    for col in skewed_columns:
        if col in X.columns:
            X[col] = np.log1p(X[col])

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X_scaled, X, scaler


X_scaled, X_model, scaler = prepare_clustering_matrix(customer_rfm, CLUSTER_FEATURES)

silhouette_scores = {}
inertias = {}

for k in K_RANGE:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(X_scaled)

    inertias[k] = float(model.inertia_)
    silhouette_scores[k] = float(silhouette_score(X_scaled, labels))

best_k = max(silhouette_scores, key=silhouette_scores.get)

print("Silhouette scores:")
for k, score in silhouette_scores.items():
    print(f"k={k}: {score:.4f}")

print(f"Số cụm được chọn: k={best_k}")

## 7. Huấn luyện K-Means và xuất kết quả

In [ ]:
final_kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
customer_rfm["cluster"] = final_kmeans.fit_predict(X_scaled)

# PCA dùng để vẽ 2D trên app hoặc trong báo cáo
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_result = pca.fit_transform(X_scaled)
customer_rfm["pca_1"] = pca_result[:, 0]
customer_rfm["pca_2"] = pca_result[:, 1]

cluster_profile = (
    customer_rfm.groupby("cluster")[CLUSTER_FEATURES]
    .mean()
    .round(2)
    .reset_index()
)

cluster_counts = (
    customer_rfm["cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .reset_index(name="customer_count")
)

cluster_profile = cluster_profile.merge(cluster_counts, on="cluster", how="left")
cluster_profile

## 8. Đặt tên cụm

Tên cụm có thể chỉnh thủ công sau khi xem `cluster_profile`. Notebook sẽ tự tạo tên gợi ý để app dễ hiển thị.

In [ ]:
def assign_cluster_names(profile: pd.DataFrame) -> dict:
    """Gợi ý tên cụm dựa trên recency, frequency và monetary."""
    profile = profile.copy()
    profile["value_score"] = (
        profile["monetary"].rank(ascending=True)
        + profile["frequency"].rank(ascending=True)
        + profile["recency"].rank(ascending=False)
    )

    cluster_names = {int(row["cluster"]): "Khách hàng tiềm năng" for _, row in profile.iterrows()}

    best_cluster = int(profile.sort_values("value_score", ascending=False).iloc[0]["cluster"])
    weak_cluster = int(profile.sort_values("value_score", ascending=True).iloc[0]["cluster"])
    inactive_cluster = int(profile.sort_values("recency", ascending=False).iloc[0]["cluster"])

    cluster_names[best_cluster] = "Khách hàng giá trị cao"
    cluster_names[weak_cluster] = "Khách hàng giá trị thấp"

    if inactive_cluster not in [best_cluster, weak_cluster]:
        cluster_names[inactive_cluster] = "Khách hàng ít hoạt động"

    return cluster_names


cluster_name_map = assign_cluster_names(cluster_profile)
customer_rfm["cluster_name"] = customer_rfm["cluster"].map(cluster_name_map)
cluster_profile["cluster_name"] = cluster_profile["cluster"].map(cluster_name_map)

customer_rfm.to_csv(CLUSTER_OUTPUT_PATH, index=False)

print(f"Đã xuất file: {CLUSTER_OUTPUT_PATH}")
cluster_profile

## 9. Lưu file metrics

File `model_metrics.json` dùng cho app, README hoặc phần kết quả trong báo cáo.

In [ ]:
metrics = {
    "iceberg_cube": {
        "algorithm": "BUC top-down",
        "dimensions": BUC_DIMENSIONS,
        "min_sup": MIN_SUP,
        "input_rows": int(len(cube_df)),
        "output_cells": int(len(iceberg_cube)),
        "output_file": str(ICEBERG_OUTPUT_PATH),
    },
    "clustering": {
        "algorithm": "K-Means",
        "features": CLUSTER_FEATURES,
        "k_range": list(K_RANGE),
        "best_k": int(best_k),
        "silhouette_scores": {str(k): v for k, v in silhouette_scores.items()},
        "inertias": {str(k): v for k, v in inertias.items()},
        "pca_explained_variance_ratio": [float(v) for v in pca.explained_variance_ratio_],
        "cluster_names": {str(k): v for k, v in cluster_name_map.items()},
        "output_file": str(CLUSTER_OUTPUT_PATH),
    },
    "cluster_profile": cluster_profile.to_dict(orient="records"),
}

with open(METRICS_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4, ensure_ascii=False)

print(f"Đã xuất file: {METRICS_OUTPUT_PATH}")
metrics